# MACHINE LEARNING 2026 - FINAL PROJECT (Option 1)

Course 2025/26: *C. Sun*, *M. Pavan*, *P. Zanuttigh*  

**Group Members:**        
*`Tommaso Dambi ID:2141118`*         
*`Alessandro Costabile ID:2146702`*        
*`Carlo Toffoli ID:2144522`*

## Heart Disease Analysis Using Clustering and Classification
<center>
    <img src="data/dataset-cover.jpg" style = "width: 50%;">
</center>

The dataset labels are the following:

|   id| age   | sex | dataset | cp | trestbps | chol | fbs | restecg | thalch | exang | oldpeak |slope | ca | thal | num |
| :-: | :-:  |  :-: | :-: | :-:| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| 1  |  63   | Male | Cleveland | typical angina | 145 | 233 | TRUE | Iv hypertrophy | 150 | FALSE | 2.3 | downsloping | 0 | fixed defect | 0 |
| 2  | 67   | Male | Cleveland | asymptomatic | 160 | 286 | FALSE | Iv hypertrophy | 108 | TRUE | 1.5 | flat | 3 | normal | 1 |
| 3   | 67 | Male | Cleveland | asymptomatic | 120 | 229 | FALSE | Iv hypertrophy | 129 | TRUE | 2.6 | flat | 2 | reversable | 2 |
| 4   | 37| Male | Cleveland | non-anginal | 130 | 250 | FALSE | normal | 187 | FALSE | 3.5 | downsloping | 0 | normal | 3 |

Description of the features:
 - `cp`: chest pain type:
    1. typical angina
    2. atypical angina
    3. non-anginal pain
    4. asymptomatic
 - `trestbps`: resting blood pressure (in mm Hg on admission to the hospital)
 - `chol`: serum cholestoral [mg/dL]
 - `fbs`: fasting blood sugar (True if above 120 mg/dL)
 - `ca`: number of major vessels (0-3) colored by flourosopy

Some records lack of data of some features such as `trestbps`, `chol`, `fbs`, `thalch`, `exang`, `oldpeak`, `ca`. This has to be addressed, maybe with clustering?

# Documentation used
- [Numpy](https://numpy.org/doc/stable/reference/index.html#reference)
- [Pandas](https://pandas.pydata.org/docs/reference/index.html#api)
- [Scikit-learn](https://scikit-learn.org/stable/api/index.html)

# ⚠️ OCIO
Il dataset non è incluso per ovvi motivi nel repository github. Scaricalo, estrailo e mettilo nella cartella `/data`. Ricorda di aggiornare il nome del file!

# CODE

## Imports

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tdc_fp.preproc import (
    load_dataset, split_dataset, filter_dataset, 
    encode_feature, standardize, binarize
)
# from sklearn.model_selection import train_test_split
# from sklearn.cluster import KMeans
# from sklearn.decomposition import PCA
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report, accuracy_score

## 1. Data Preprocessing
As listed below, we have written a few helper functions that will simplify the preprocessing of the dataset.

In [2]:
# TODO: Dropping columns when loading the dataset isn't ideal, move it to another function
# (we will use it to control the number of features)
csv_dataframe = load_dataset('tdc_fp/data/heart_disease_uci.csv', drop_columns=['id', 'ca']) # tried dropping `ca` because 611 samples were missing it

# print(csv_dataframe.head())

dataset, value_map = encode_feature(csv_dataframe)

# Print the mapped array
print("Dataset is:")
print(dataset.dtype, dataset.shape)

# TODO: deal with nan values BEFORE STANDARDIZATION (issue #2)
nan_occurencies = [(col_head, int(np.sum(np.isnan(col)))) for col, col_head in zip(dataset.T, csv_dataframe.columns)]
print("\nFeatures with missing values (and the count of samples that miss them)")
print({i[0]:i[1] for i in nan_occurencies if i[1]})

dataset_filtered = filter_dataset(dataset)
print(f"\nTrashed samples (`ca` feature neglected): {dataset.shape[0] - dataset_filtered.shape[0]}\n")

X, Y = split_dataset(dataset_filtered)
print("X and Y shapes after dropping features and filtering:")
print(X.shape, Y.shape)

# Standardization
# TODO: Ask the teacher whether this is fine!
X = standardize(X)

# Comment this code if you're trying to do multiclass classification
Y = binarize(Y)

print(f"\nUnique values of Y: {np.unique(Y)}")
print(pd.DataFrame(X).describe().round(3))

Dataset is:
float64 (920, 14)

Features with missing values (and the count of samples that miss them)
{'trestbps': 59, 'chol': 30, 'fbs': 90, 'thalch': 55, 'exang': 55, 'oldpeak': 62}

Trashed samples (`ca` feature neglected): 179

X and Y shapes after dropping features and filtering:
(741, 13) (741, 1)

Unique values of Y: [0 1]
            0        1        2        3        4        5        6        7   \
count  741.000  741.000  741.000  741.000  741.000  741.000  741.000  741.000   
mean    -0.000   -0.000    0.000    0.000    0.000   -0.000    0.000    0.000   
std      1.001    1.001    1.001    1.001    1.001    1.001    1.001    1.001   
min     -2.671   -0.554   -0.925   -1.811   -7.154   -2.355   -0.420   -1.471   
25%     -0.756   -0.554   -0.925   -0.658   -0.688   -0.248   -0.420    0.113   
50%      0.096   -0.554   -0.002   -0.658   -0.149    0.115   -0.420    0.113   
75%      0.734   -0.554   -0.002    0.495    0.390    0.543   -0.420    0.113   
max      2.544    1.

## 2. Cluster analisys